# slm-rag-bench — T4 smoke ladder (all-in-one)

Clones the repo, builds the **locked** environment (`uv sync --frozen --extra serve`), and runs the
README GPU ladder on a free Colab **T4**: CPU self-check → judge probe → one capped CRAG cell
(`crag-qwen2.5-1.5b`) → optional `--n 30` pilot.

| Fact | Value |
|---|---|
| Cost | $0 (free T4; `gpu_usd_per_hour` stays the pricing.yaml placeholder — smoke cells are not cost measurements) |
| VRAM peak | ~8.5 / 16 GB — vLLM budget 0.5×16=8 GB (Qwen2.5-1.5B-AWQ ~1.2 GB weights; Mistral-7B-AWQ ~4 GB, **sequential**, never concurrent) + MiniLM 0.1 GB + flan-t5 0.3 GB co-resident |
| AWQ on T4 | T4 = sm_75, exactly the plan's AWQ floor; `--enforce-eager` is hardcoded in `src/runner/serve.py` |
| HF token | not needed — Qwen2.5-1.5B-AWQ and solidrust/Mistral-7B-AWQ are public (llama3.1-8b is NOT in this ladder) |
| Wall clock | ~30–45 min for cells 1–5 (installs + downloads dominate); optional cell 6 adds ~25–35 min |

**Fresh-import ready.** Designed to be opened straight from GitHub on a new runtime —
cell 2 clones into `/content/slm-rag-bench` by absolute path, pulls latest `main`, and can be
re-run safely (no nested clones). Proven 2026-11-07: smoke 18/19 (capped) and pilot 19/19 (full
corpus) both green on a free T4. Results land in `{REPO_DIR}/results/` — save them (cell 8)
before the VM dies.

**What this proves / does not prove.** It proves vLLM starts, streaming+usage works (the cost axis),
the judge path (ragas collections) works, and the full chain runs end-to-end. It does NOT produce a
publishable cell: the capped corpus inflates every retrieval metric (recorded in the manifest), and
n=5 is far below n=300. Expect **18/19** on this capped smoke: the cap drops every sampled query's gold, so all
retrieval scores are coverage-only zeros and the `contract: retrieval scores carry metadata`
gate red-flags them — correct behavior, the same warning run_cell prints (uncapped runs abort
instead, so that red is reachable only on capped runs). 19/19 is possible only in cell 6, the
uncapped pilot. The notebook parses gates instead of failing on rc=1.


In [ ]:
#@title 1. Configure + GPU check
REPO_URL = "https://github.com/Khoa-NguyenAnh-Hoang/slm-rag-bench.git"  # @param {type:"string"}
MODEL_KEY = "qwen2.5-1.5b"  # @param ["qwen2.5-1.5b", "qwen3-4b"]
N_QUERIES = 5           # smoke size; the pilot cell uses 30
CORPUS_CAP = 20000      # capped corpus = fast smoke, NOT publishable (manifest records the cap)

import torch
assert torch.cuda.is_available(), "No GPU — select Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
print(f"{name} | cc={cc} | sm_75+ needed for AWQ: {cc >= (7, 5)} | torch {torch.__version__}")
assert cc >= (7, 5), f"AWQ needs sm_75+; this card is {cc}"


In [ ]:
#@title 2. Clone repo + build the locked environment (~5–10 min)
import os, subprocess, time

def sh(cmd, allow_fail=False):
    t = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout or "")[-2000:] + (("\n" + r.stderr[-2000:]) if r.returncode else ""), flush=True)
    if r.returncode and not allow_fail:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    print(f"[{time.time() - t:.0f}s] rc={r.returncode}: {cmd}", flush=True)
    return r.returncode

# Absolute paths: re-running this cell can never nest a clone inside a clone.
REPO_DIR = "/content/slm-rag-bench"
if not os.path.isdir(REPO_DIR):
    sh(f"git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("git pull")                       # fresh clone = latest; existing clone = update to latest
sh("git log --oneline -1")           # the code revision this session runs (also in manifests)
sh("pip install -q uv")
# --frozen = install exactly uv.lock (cross-platform lock; uv fetches CPython 3.12 itself,
# isolated from Colab's system torch — no runtime restart needed).
sh("uv sync --frozen --extra serve")


In [ ]:
#@title 3. CPU self-check — 9/9 must pass (~3–6 min; downloads HotpotQA dev + MiniLM + flan-t5)
# Validates the venv before any paid-in-time GPU step. Cheap-before-expensive by design.
sh("uv run python -m tools.selfcheck")


In [ ]:
#@title 4. Judge probe — generator + judge servers sequentially (~8–12 min)
# Pulls Qwen2.5-1.5B-AWQ (~1.2 GB) and solidrust/Mistral-7B-AWQ (~4.4 GB), then proves the two
# gates everything depends on: streaming usage block (the cost axis) and ragas faithfulness >= 4/5.
# A failure HERE is a blocker — fix before running any cell (README ladder step 2).
sh("uv run python -m src.runner.judge_probe --config configs/experiment.yaml --model-key " + MODEL_KEY)


In [ ]:
#@title 5. SMOKE CELL: crag-qwen2.5-1.5b n=5, capped corpus (~10–15 min; first run downloads the train split ~590 MB)
# rc=1 with red gates is EXPECTED if the flan-t5 proxy labels everything Correct (known validity
# finding) — the scorecard + manifest are still written and parsed below.
rc = sh(f"uv run python -m src.runner.run_cell --only crag-{MODEL_KEY} "
        f"--n {N_QUERIES} --limit-corpus {CORPUS_CAP}", allow_fail=True)

import json, pathlib
cell_dir = pathlib.Path("results") / f"crag-{MODEL_KEY}"
manifest = json.loads((cell_dir / "manifest.json").read_text(encoding="utf-8"))
print("\n== manifest ==", manifest["git_rev"], "|", manifest["hardware"]["gpu"],
      "| corpus", manifest["corpus"]["paragraphs"], "paras | cap", manifest["corpus"]["max_paras"])
print("== deviations ==")
for d in manifest["deviations"]:
    print(" -", d)
card = json.loads((cell_dir / "scorecard.json").read_text(encoding="utf-8"))
print(f"\n== gates {card.get('gates_green')} ==")
for k, v in (card.get("gates") or {}).items():
    print(("PASS  " if v else "FAIL  ") + k)
print("\nRED gates are the story to read; a missing scorecard (no gates key) means a crash —",
      "read the traceback above.")


In [ ]:
#@title 6. (Optional) pilot: n=30, FULL 483,696-paragraph corpus (~25–35 min)
# This is README ladder step 4 / plan §5.3 Stage 2: the real per-cell number.
# Multiply its wall-clock by 10 to price the 6-cell rental (plan §5.2).
# RAM note: the full train split loads ~4–6 GB in Colab's ~12.7 GB — close the other notebooks.
RUN_PILOT = False  # @param {type:"boolean"}
if RUN_PILOT:
    sh(f"uv run python -m src.runner.run_cell --only crag-{MODEL_KEY} --n 30", allow_fail=True)
else:
    print("skipped (RUN_PILOT=False)")


In [ ]:
#@title 7. Validate the artifacts
import json, pathlib
for d in sorted(pathlib.Path("results").glob("*/")):
    mf, cf = d / "manifest.json", d / "scorecard.json"
    if not mf.exists():
        print(f"{d.name}: NO MANIFEST — died before provenance, do not trust anything here")
        continue
    m = json.loads(mf.read_text(encoding="utf-8"))
    s = json.loads(cf.read_text(encoding="utf-8")) if cf.exists() else {}
    print(f"{d.name}: n={m['n']} corpus={m['corpus']['paragraphs']} cap={m['corpus']['max_paras']} "
          f"gates={s.get('gates_green', '—')} wall={s.get('wall_clock_s', 0)/60:.1f} min")
    q = d / "queries.jsonl"
    print(f"  queries.jsonl: {sum(1 for _ in q.open(encoding='utf-8')) if q.exists() else 0} records (committed artifact)")


In [ ]:
#@title 8. (Optional) copy results/ to Google Drive + commit reminder
SAVE_TO_DRIVE = False  # @param {type:"boolean"}
if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/slm-rag-bench-results"
    shutil.copytree("results", dst, dirs_exist_ok=True)
    print("copied ->", dst)
print("Results are the entire deliverable: commit results/ to GitHub (git add results/ && git"
      " commit && git push) or download the folder — a reclaimed Colab VM takes it with it.")
